# Series df[series] 

## OneDrive Data Access Setup

This notebook demonstrates how to load data from OneDrive using Microsoft Graph API.

### Step 1: Import Libraries

In [1]:
from azure.identity import ClientSecretCredential
from dotenv import load_dotenv
from io import BytesIO
import pandas as pd
import requests
import os

### Step 2: Authenticate with Microsoft Graph

Load credentials from `.env` file and get access token.

In [2]:
# Load credentials
load_dotenv()
CLIENT_ID = os.getenv("SHAREPOINT_CLIENT_ID")
CLIENT_SECRET = os.getenv("SHAREPOINT_CLIENT_SECRET")
TENANT_ID = os.getenv("SHAREPOINT_TENANT_ID")

# Create credential and get access token
credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
access_token = credential.get_token("https://graph.microsoft.com/.default").token

print("✅ Connected to Microsoft Graph")

✅ Connected to Microsoft Graph


### Step 3: Configure OneDrive Settings

In [3]:
# Your OneDrive email and base folder
USER_EMAIL = "info@adeyanjuteslim.co.uk"
BASE_FOLDER = "TeslimDataLakes"  # Main folder in OneDrive

### Step 4: Define Helper Functions

Functions to list files and load different file formats from OneDrive.

In [4]:
def load_csv_from_onedrive(subfolder: str, filename: str) -> pd.DataFrame:
    """
    Load a CSV file from OneDrive into a pandas DataFrame.
    
    Args:
        subfolder: Subfolder path inside BASE_FOLDER (e.g., "MyData/2024")
        filename: CSV filename (e.g., "data.csv")
    
    Returns:
        pandas DataFrame
    """
    file_path = f"{BASE_FOLDER}/{subfolder}/{filename}"
    url = f"https://graph.microsoft.com/v1.0/users/{USER_EMAIL}/drive/root:/{file_path}:/content"
    
    response = requests.get(url, headers={"Authorization": f"Bearer {access_token}"})
    response.raise_for_status()
    
    return pd.read_csv(BytesIO(response.content))

In [5]:
def load_json_from_onedrive(subfolder: str, filename: str) -> pd.DataFrame:
    """
    Load a JSON file from OneDrive into a pandas DataFrame.
    Handles different JSON structures automatically.
    
    Args:
        subfolder: Subfolder path inside BASE_FOLDER
        filename: JSON filename (e.g., "data.json")
    
    Returns:
        pandas DataFrame
    """
    import json
    
    file_path = f"{BASE_FOLDER}/{subfolder}/{filename}"
    url = f"https://graph.microsoft.com/v1.0/users/{USER_EMAIL}/drive/root:/{file_path}:/content"
    
    response = requests.get(url, headers={"Authorization": f"Bearer {access_token}"})
    response.raise_for_status()
    
    # Try to parse JSON and handle different structures
    try:
        # First try standard pandas read_json
        return pd.read_json(BytesIO(response.content))
    except ValueError:
        # If that fails, load as dict and convert
        json_data = json.loads(response.content)
        
        # If it's a dict with scalar values, convert to single-row DataFrame
        if isinstance(json_data, dict) and all(not isinstance(v, (list, dict)) for v in json_data.values()):
            return pd.DataFrame([json_data])
        
        # Otherwise try converting the dict to DataFrame
        return pd.DataFrame(json_data)

In [6]:
def load_excel_from_onedrive(subfolder: str, filename: str, sheet_name: str = 0) -> pd.DataFrame:
    """
    Load an Excel file from OneDrive into a pandas DataFrame.
    
    Args:
        subfolder: Subfolder path inside BASE_FOLDER
        filename: Excel filename (e.g., "data.xlsx" or "data.xls")
        sheet_name: Sheet name or index (default: 0 for first sheet)
    
    Returns:
        pandas DataFrame
    """
    file_path = f"{BASE_FOLDER}/{subfolder}/{filename}"
    url = f"https://graph.microsoft.com/v1.0/users/{USER_EMAIL}/drive/root:/{file_path}:/content"
    
    response = requests.get(url, headers={"Authorization": f"Bearer {access_token}"})
    response.raise_for_status()
    
    return pd.read_excel(BytesIO(response.content), sheet_name=sheet_name)

In [7]:
def load_parquet_from_onedrive(subfolder: str, filename: str) -> pd.DataFrame:
    """
    Load a Parquet file from OneDrive into a pandas DataFrame.
    
    Args:
        subfolder: Subfolder path inside BASE_FOLDER
        filename: Parquet filename (e.g., "data.parquet")
    
    Returns:
        pandas DataFrame
    """
    file_path = f"{BASE_FOLDER}/{subfolder}/{filename}"
    url = f"https://graph.microsoft.com/v1.0/users/{USER_EMAIL}/drive/root:/{file_path}:/content"
    
    response = requests.get(url, headers={"Authorization": f"Bearer {access_token}"})
    response.raise_for_status()
    
    return pd.read_parquet(BytesIO(response.content))

In [8]:
def load_file_from_onedrive(subfolder: str, filename: str, **kwargs) -> pd.DataFrame:
    """
    Smart function that auto-detects file type and loads it.
    
    Args:
        subfolder: Subfolder path inside BASE_FOLDER
        filename: Filename (e.g., "data.csv", "data.json", "data.xlsx")
        **kwargs: Additional arguments passed to pandas reader (e.g., sheet_name for Excel)
    
    Returns:
        pandas DataFrame
    """
    file_extension = filename.lower().split('.')[-1]
    
    if file_extension == 'csv':
        return load_csv_from_onedrive(subfolder, filename)
    elif file_extension == 'json':
        return load_json_from_onedrive(subfolder, filename)
    elif file_extension in ['xlsx', 'xls']:
        return load_excel_from_onedrive(subfolder, filename, **kwargs)
    elif file_extension == 'parquet':
        return load_parquet_from_onedrive(subfolder, filename)
    else:
        raise ValueError(f"Unsupported file type: .{file_extension}")

In [9]:
def list_files(subfolder: str):
    """
    List all files in an OneDrive folder.
    
    Args:
        subfolder: Subfolder path inside BASE_FOLDER
    
    Returns:
        List of file dictionaries
    """
    folder_path = f"{BASE_FOLDER}/{subfolder}"
    url = f"https://graph.microsoft.com/v1.0/users/{USER_EMAIL}/drive/root:/{folder_path}:/children"
    
    response = requests.get(url, headers={"Authorization": f"Bearer {access_token}"})
    response.raise_for_status()
    
    files = response.json().get('value', [])
    print(f"📁 Files in {folder_path}:")
    for file in files:
        print(f"   • {file['name']}")
    
    return files

---
### Usage Examples

#### Example 1: List files in a folder

In [10]:
files = list_files("FinancialTransactionsDatasetAnalytics(Kaggle)")

📁 Files in TeslimDataLakes/FinancialTransactionsDatasetAnalytics(Kaggle):
   • cards_data.csv
   • mcc_codes.json
   • train_fraud_labels.json
   • users_data.csv


In [11]:
file2 = list_files("SundryDataSources")

📁 Files in TeslimDataLakes/SundryDataSources:
   • 22025_QS_World_University_Rankings.csv
   • Apple-Amazon-Google-Microsoft-DailyStockReturns-10years.xlsx
   • Country Economy Indicator 1950 - 2022.xlsx
   • Country GDP-perCapital 1952 - 2007.xlsx
   • End-of-Month Retail Inventories and Inventories1992-2024.xlsx
   • Job Roles Opening Analyst.xlsx
   • Monthly Retail and Food Services Sales by Kind of Business1992-2025.xlsx
   • Motor Fuel_Efficiency_mpg.xlsx
   • Retail Transaction Loyalty Discount Report.xlsx
   • Skilled Worker Employer Sponsorship List.xlsx
   • Time Series Data.xlsx
   • UK Constituency Election Result 2019 and 2024.xlsx
   • UK Election Result Poll.xlsx
   • UK Smoking Data.csv
   • UKHouse of Commons General Election 2024 results by constituency.xlsx


#### Example 2: Load different file types

In [67]:
# Load CSV file
users = load_csv_from_onedrive(
    "FinancialTransactionsDatasetAnalytics(Kaggle)", 
    "users_data.csv"
)

# Load another CSV
cards = load_csv_from_onedrive(
    "FinancialTransactionsDatasetAnalytics(Kaggle)", 
    "cards_data.csv"
)

# Load JSON file
mcc = load_json_from_onedrive(
    "FinancialTransactionsDatasetAnalytics(Kaggle)", 
    "mcc_codes.json"
)

print(f"✅ Loaded users (CSV): {users.shape}")
print(f"✅ Loaded cards (CSV): {cards.shape}")
print(f"✅ Loaded MCC codes (JSON): {mcc.shape}")

✅ Loaded users (CSV): (2000, 14)
✅ Loaded cards (CSV): (6146, 13)
✅ Loaded MCC codes (JSON): (1, 109)


In [13]:
UK_Smoking_Data = load_csv_from_onedrive(
    "SundryDataSources",
    "UK Smoking Data.csv"
)   

In [14]:
UK_Smoking_Data

,gender,age,marital_status,highest_qualification,nationality,ethnicity,gross_income,region,smoke,amt_weekends,amt_weekdays,type
0,Male,38,Divorced,No Qualification,British,White,"2,600 to 5,200",The North,No,NaN,NaN,NaN
1,Female,42,Single,No Qualification,British,White,"Under 2,600",The North,Yes,12.0,12.0,Packets
2,Male,40,Married,Degree,English,White,"28,600 to 36,400",The North,No,NaN,NaN,NaN
3,Female,40,Married,Degree,English,White,"10,400 to 15,600",The North,No,NaN,NaN,NaN
4,Female,39,Married,GCSE/O Level,British,White,"2,600 to 5,200",The North,No,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1686,Male,22,Single,No Qualification,Scottish,White,"2,600 to 5,200",Scotland,No,NaN,NaN,NaN
1687,Female,49,Divorced,Other/Sub Degree,English,White,"2,600 to 5,200",Scotland,Yes,20.0,20.0,Hand-Rolled
1688,Male,45,Married,Other/Sub Degree,Scottish,White,"5,200 to 10,400",Scotland,No,NaN,NaN,NaN
1689,Female,51,Married,No Qualification,English,White,"2,600 to 5,200",Scotland,Yes,20.0,20.0,Packets


#### Example 3: Auto-detect file type

In [68]:
# Smart function that detects file type automatically
# Works with .csv, .json, .xlsx, .xls, .parquet

# TEMPLATE - Replace "YourFolder" with your actual folder name
# Uncomment the lines below when you're ready to use them

# data1 = load_file_from_onedrive("YourFolder", "data.csv")
# data2 = load_file_from_onedrive("YourFolder", "data.json")
# data3 = load_file_from_onedrive("YourFolder", "data.xlsx", sheet_name="Sheet1")

# WORKING EXAMPLE with actual data:
fraud_labels = load_file_from_onedrive(
    "FinancialTransactionsDatasetAnalytics(Kaggle)",
    "train_fraud_labels.json"
)

print(f"✅ Loaded fraud labels: {fraud_labels.shape}")
fraud_labels.head()

✅ Loaded fraud labels: (8914963, 1)


,target
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No


In [69]:
# Preview the data
users.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [70]:
cards.head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


---
### Quick Reference

**Supported File Types:**
- ✅ CSV (`.csv`)
- ✅ JSON (`.json`)
- ✅ Excel (`.xlsx`, `.xls`)
- ✅ Parquet (`.parquet`)

**Usage Examples:**

```python
# 1. Run Steps 1-4 above once

# 2. List files in a folder
list_files("your/subfolder")

# 3. Load specific file types
df_csv = load_csv_from_onedrive("subfolder", "data.csv")
df_json = load_json_from_onedrive("subfolder", "data.json")
df_excel = load_excel_from_onedrive("subfolder", "data.xlsx", sheet_name="Sheet1")
df_parquet = load_parquet_from_onedrive("subfolder", "data.parquet")

# 4. Smart auto-detect (easiest method!)
df = load_file_from_onedrive("subfolder", "data.csv")  # Works with any supported type
```

**Configuration:**
- Update `USER_EMAIL` with your OneDrive email
- Update `BASE_FOLDER` with your main folder name
- Credentials stored in `.env` file

### Additional File Types

You can easily add support for other pandas-supported formats by following the same pattern:

**Other pandas readers available:**
- `pd.read_sql()` - SQL databases
- `pd.read_pickle()` - Pickle files
- `pd.read_feather()` - Feather format
- `pd.read_hdf()` - HDF5 format
- `pd.read_stata()` - Stata files
- `pd.read_sas()` - SAS files

Just follow the pattern above and replace `pd.read_csv()` with the appropriate reader!

In [8]:
import pandas as pd
import seaborn as sns

# Load Titanic dataset
df = sns.load_dataset('titanic')
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [10]:
df['age'].where(df['age'] <= 50, other=5000)

0        22.0
1        38.0
2        26.0
3        35.0
4        35.0
        ...  
886      27.0
887      19.0
888    5000.0
889      26.0
890      32.0
Name: age, Length: 891, dtype: float64

In [12]:
df['fare']

0       7.2500
1      71.2833
2       7.9250
3      53.1000
4       8.0500
        ...   
886    13.0000
887    30.0000
888    23.4500
889    30.0000
890     7.7500
Name: fare, Length: 891, dtype: float64

In [14]:
df['nnn'] = df['age'].where(df['fare'] <= 10, other="check")

In [15]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,nnn
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,22.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,check
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,26.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,check
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True,check
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True,check
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False,check
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True,check


In [17]:
df['age'].mask(df['age'] > 18, other='Minor').head(10)

0    Minor
1    Minor
2    Minor
3    Minor
4    Minor
5      NaN
6    Minor
7      2.0
8    Minor
9     14.0
Name: age, dtype: object

In [ ]:
df['age'].where()